# 模型 6：BiLSTM + Attention

**负责人：** 6 号　**阶段：** 3 模型

token 序列 → BiLSTM + 加性注意力（从零训练）。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [ ] 输入同模型 5。5 号的 `tokens_multi` 还没交；网络已经按那份输入来写（float 序列加 mask），文件到了可以直接换，不用改结构
- [x] FastText 输入要用对应折的版本：第 k 折用 `_fold{k}`，预测测试集用 `_full`
- [x] 用 masking 或 packed sequence 处理 padding；1–2 层，hidden 128
- [x] 挑几条推文画注意力热力图
- [x] 每次实验先用 `new_run_id()` 生成 run_id，评分记录和预测文件都带上同一个 run_id

## 产出

- `results/m6_bilstm.csv`
- 用 `save_preds(run_id, ...)` 保存 OOF 概率和测试集概率（存到 `DATA_ROOT/preds/`）
- `figures/m6_attention_*.png`

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

## Approach

Each tweet is a sequence of FastText token vectors, `fasttext_tok_fold{k}` for fold k and `fasttext_tok_full` for the test refit. Padding is skipped with `pack_padded_sequence`, then a bidirectional LSTM (hidden 128 each direction) and additive attention. The attention mask is applied before the softmax, so padding positions get no weight.

Depth 1 and depth 2 are both trained. Early stopping watches a 15% group holdout taken from the training folds, not the fold that is being scored, so the out-of-fold macro-F1 does not pick the epoch with the validation labels. The test split is scored once, on the better depth. Its epoch count is the median best epoch from the five folds.

Focal loss is still optional and is not part of this run. Member 5's frozen token sequences can replace the FastText sequences later without changing the network: the input is already `(batch, length, dim)` plus a mask.

In [ ]:
from Member6_file.m6_bilstm import run_bilstm

results = run_bilstm()
results.loc[:, ["split", "macro_f1", "accuracy", "run_id", "params"]]